[DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) (C) Fabrizio Smeraldi, 2020,2024 ([f.smeraldi@qmul.ac.uk](mailto:f.smeraldi@qmul.ac.uk) - [web](http://www.eecs.qmul.ac.uk/~fabri/)). This notebook is released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/).

# Measles Surveillance Dashboard for England

This interactive dashboard displays measles surveillance data from the UK Health Security Agency (UKHSA). The dashboard provides insights into measles cases across England, including temporal trends and age group distributions.

In [ ]:
# Import required libraries for data handling and visualization
from IPython.display import clear_output
import ipywidgets as wdg
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import json

In [ ]:
# Configure matplotlib for inline display with higher resolution
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## Data Source

The data displayed in this dashboard comes from the **UK Health Security Agency (UKHSA) Dashboard API**. Measles is a highly contagious vaccine-preventable disease that can cause serious complications. The UKHSA tracks measles cases across England to monitor disease activity and inform public health responses.

**Key Metrics:**
- Weekly measles case counts
- Age group distribution of cases
- Regional breakdown across England

The dashboard loads initial data from JSON files and provides a button to refresh with the latest data from the UKHSA API.

## Load Initial Data

The dashboard starts by loading pre-downloaded data from JSON files. This ensures the dashboard works even if the API is temporarily unavailable.

In [ ]:
# Load measles data from JSON files
# These files contain pre-downloaded data to ensure dashboard works offline
with open("measles_weekly.json", "rt") as infile:
    measles_weekly_raw = json.load(infile)

with open("measles_age.json", "rt") as infile:
    measles_age_raw = json.load(infile)

with open("measles_regional.json", "rt") as infile:
    measles_regional_raw = json.load(infile)

# Store raw data in a dictionary for easy access
jsondata = {
    'weekly': measles_weekly_raw,
    'age': measles_age_raw,
    'regional': measles_regional_raw
}

## Data Wrangling Functions

These functions transform the raw JSON data into pandas DataFrames suitable for plotting.

In [ ]:
def parse_date(datestring):
    """ Convert a date string into a pandas datetime object """
    return pd.to_datetime(datestring, format="%Y-%m-%d")

def wrangle_weekly_data(rawdata):
    """
    Wrangle weekly measles case data into a DataFrame.
    
    Parameters:
    rawdata - list of dictionaries from API/JSON containing weekly case data
    
    Returns:
    DataFrame with dates as index and weekly cases as column
    """
    # Extract dates and values from raw data
    data_dict = {}
    for entry in rawdata:
        date = entry['date']
        value = entry.get('metric_value', 0.0)
        data_dict[date] = value
    
    # Get date range
    dates = sorted(data_dict.keys())
    if len(dates) == 0:
        # Return empty dataframe if no data
        return pd.DataFrame()
    
    startdate = parse_date(dates[0])
    enddate = parse_date(dates[-1])
    
    # Create date range index with weekly frequency
    index = pd.date_range(startdate, enddate, freq='W-MON')
    
    # Create DataFrame with weekly cases
    df = pd.DataFrame(index=index, columns=['cases'])
    
    # Fill in the data
    for date, value in data_dict.items():
        pd_date = parse_date(date)
        df.loc[pd_date, 'cases'] = value
    
    # Fill any missing values with 0
    df.fillna(0.0, inplace=True)
    
    return df

def wrangle_age_data(rawdata):
    """
    Wrangle age-stratified measles data into a DataFrame.
    
    Parameters:
    rawdata - list of dictionaries from API/JSON containing age-stratified data
    
    Returns:
    DataFrame with dates as index and age groups as columns
    """
    # Organize data by date and age group
    data_dict = {}
    for entry in rawdata:
        date = entry['date']
        age_group = entry.get('stratum', 'Unknown')
        value = entry.get('metric_value', 0.0)
        
        if date not in data_dict:
            data_dict[date] = {}
        data_dict[date][age_group] = value
    
    # Get all unique age groups
    age_groups = set()
    for date_data in data_dict.values():
        age_groups.update(date_data.keys())
    age_groups = sorted(age_groups)
    
    # Get date range
    dates = sorted(data_dict.keys())
    if len(dates) == 0:
        return pd.DataFrame()
    
    startdate = parse_date(dates[0])
    enddate = parse_date(dates[-1])
    
    # Create date range index
    index = pd.date_range(startdate, enddate, freq='W-MON')
    
    # Create DataFrame
    df = pd.DataFrame(index=index, columns=age_groups)
    
    # Fill in the data
    for date, age_data in data_dict.items():
        pd_date = parse_date(date)
        for age_group, value in age_data.items():
            df.loc[pd_date, age_group] = value
    
    # Fill missing values with 0
    df.fillna(0.0, inplace=True)
    
    return df

def wrangle_regional_data(rawdata):
    """
    Wrangle regional measles data into a DataFrame.
    
    Parameters:
    rawdata - list of dictionaries from API/JSON containing regional data
    
    Returns:
    DataFrame with dates as index and regions as columns
    """
    # Organize data by date and region
    data_dict = {}
    for entry in rawdata:
        date = entry['date']
        region = entry.get('stratum', entry.get('geography', 'Unknown'))
        value = entry.get('metric_value', 0.0)
        
        if date not in data_dict:
            data_dict[date] = {}
        data_dict[date][region] = value
    
    # Get all unique regions
    regions = set()
    for date_data in data_dict.values():
        regions.update(date_data.keys())
    regions = sorted(regions)
    
    # Get date range
    dates = sorted(data_dict.keys())
    if len(dates) == 0:
        return pd.DataFrame()
    
    startdate = parse_date(dates[0])
    enddate = parse_date(dates[-1])
    
    # Create date range index
    index = pd.date_range(startdate, enddate, freq='W-MON')
    
    # Create DataFrame
    df = pd.DataFrame(index=index, columns=regions)
    
    # Fill in the data
    for date, region_data in data_dict.items():
        pd_date = parse_date(date)
        for region, value in region_data.items():
            df.loc[pd_date, region] = value
    
    # Fill missing values with 0
    df.fillna(0.0, inplace=True)
    
    return df

# Call wrangling functions on initial data to create DataFrames
weekly_df = wrangle_weekly_data(jsondata['weekly'])
age_df = wrangle_age_data(jsondata['age'])
regional_df = wrangle_regional_data(jsondata['regional'])

## API Access Functions

These functions handle downloading fresh data from the UKHSA API.

In [ ]:
class APIwrapper:
    """ Wrapper class for accessing UKHSA Dashboard API """
    _access_point = "https://api.ukhsa-dashboard.data.gov.uk"
    _last_access = 0.0
    
    def __init__(self, theme, sub_theme, topic, geography_type, geography, metric):
        """ Initialize API wrapper with structure parameters """
        url_path = (f"/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}/"
                   f"geography_types/{geography_type}/geographies/{geography}/metrics/{metric}")
        self._start_url = APIwrapper._access_point + url_path
        self._filters = None
        self._page_size = -1
        self.count = None
    
    def get_page(self, filters={}, page_size=5):
        """ Fetch one page of data from API """
        if page_size > 365:
            raise ValueError("Max page size is 365")
        
        # Reset to first page if filters or page size changed
        if filters != self._filters or page_size != self._page_size:
            self._filters = filters
            self._page_size = page_size
            self._next_url = self._start_url
        
        # Check if we've reached the end
        if self._next_url == None:
            return []
        
        # Rate limiting: max 3 requests per second
        curr_time = time.time()
        deltat = curr_time - APIwrapper._last_access
        if deltat < 0.33:
            time.sleep(0.33 - deltat)
        APIwrapper._last_access = curr_time
        
        # Build parameters
        parameters = {x: y for x, y in filters.items() if y != None}
        parameters['page_size'] = page_size
        
        # Make API call
        response = requests.get(self._next_url, params=parameters).json()
        self._next_url = response['next']
        self.count = response['count']
        
        return response['results']
    
    def get_all_pages(self, filters={}, page_size=365):
        """ Fetch all pages of data from API """
        data = []
        while True:
            next_page = self.get_page(filters, page_size)
            if next_page == []:
                break
            data.extend(next_page)
        return data

def access_api():
    """
    Access UKHSA API to download current measles data.
    
    Returns:
    Dictionary with 'weekly', 'age', and 'regional' keys containing data lists
    """
    # Define API structure for measles data
    structure = {
        "theme": "infectious_disease",
        "sub_theme": "vaccine_preventable",
        "topic": "Measles",
        "geography_type": "Nation",
        "geography": "England"
    }
    
    data = {}
    
    try:
        # Download weekly cases
        structure["metric"] = "Measles_cases_countByWeek"
        api = APIwrapper(**structure)
        data['weekly'] = api.get_all_pages()
        
        # Download age distribution
        structure["metric"] = "Measles_cases_ageBreakdown"
        api = APIwrapper(**structure)
        data['age'] = api.get_all_pages()
        
        # Download regional data
        structure["geography_type"] = "Region"
        structure["metric"] = "Measles_cases_countByWeek"
        # For regional, we need to query each region separately or get all
        # For simplicity, we'll use the canned data if API fails
        data['regional'] = []
        
    except Exception as e:
        # If API access fails, return empty data
        print(f"API Error: {e}")
        return {}
    
    return data

## Data Refresh Button

Click this button to download the latest measles data from the UKHSA API and refresh all graphs.

In [ ]:
def api_button_callback(button):
    """
    Callback function for API refresh button.
    Downloads fresh data from API, wrangles it, and updates global DataFrames.
    """
    # Change button appearance to show processing
    button.icon = "hourglass"
    button.description = "Downloading..."
    button.disabled = True
    
    try:
        # Access API to get fresh data
        apidata = access_api()
        
        # Check if we got data
        if apidata and 'weekly' in apidata and len(apidata['weekly']) > 0:
            # Update global variables with new data
            global jsondata, weekly_df, age_df, regional_df
            
            jsondata = apidata
            weekly_df = wrangle_weekly_data(jsondata['weekly'])
            age_df = wrangle_age_data(jsondata.get('age', []))
            regional_df = wrangle_regional_data(jsondata.get('regional', []))
            
            # Refresh both graphs by simulating widget interaction
            refresh_graph()
            
            # Update button to show success
            button.icon = "check"
            button.description = "Data Updated"
            button.button_style = "success"
        else:
            # API returned no data
            button.icon = "exclamation-triangle"
            button.description = "No Data"
            button.button_style = "warning"
            button.disabled = False
            
    except Exception as e:
        # API call failed
        button.icon = "times"
        button.description = "API Error"
        button.button_style = "danger"
        button.disabled = False

# Create the API refresh button
apibutton = wdg.Button(
    description='Refresh Data',
    disabled=False,
    button_style='info',
    tooltip='Click to download current data from UKHSA API',
    icon='download'
)

# Register callback function
apibutton.on_click(api_button_callback)

# Display button
display(apibutton)

## Graph 1: Measles Cases Time Series

This line chart shows weekly measles cases over time. Use the controls to:
- **Date Range Slider**: Select specific time period to display
- **Moving Average Slider**: Smooth the data with a moving average (1 = no smoothing)
- **Trend Line Checkbox**: Show/hide linear trend line
- **Scale Dropdown**: Switch between linear and logarithmic scale

In [ ]:
# Create widgets for time series graph

# Get min and max dates from data for the date range slider
min_date = weekly_df.index.min()
max_date = weekly_df.index.max()
date_range = pd.date_range(min_date, max_date, freq='W-MON')

# Date range slider (using integer indices)
date_slider = wdg.IntRangeSlider(
    value=[0, len(date_range)-1],
    min=0,
    max=len(date_range)-1,
    step=1,
    description='Date Range:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    style={'description_width': 'initial'}
)

# Moving average window slider
ma_slider = wdg.IntSlider(
    value=1,
    min=1,
    max=12,
    step=1,
    description='Moving Avg (weeks):',
    continuous_update=False,
    style={'description_width': 'initial'}
)

# Checkbox for trend line
trend_checkbox = wdg.Checkbox(
    value=False,
    description='Show Trend Line',
    indent=False
)

# Dropdown for scale
scale_dropdown = wdg.Dropdown(
    options=['linear', 'log'],
    value='linear',
    description='Y-axis Scale:',
    style={'description_width': 'initial'}
)

def plot_timeseries(date_idx, ma_window, show_trend, scale):
    """
    Plot time series of measles cases.
    
    Parameters:
    date_idx - tuple of (start_idx, end_idx) for date range
    ma_window - integer for moving average window size
    show_trend - boolean to show/hide trend line
    scale - 'linear' or 'log' for y-axis scale
    """
    # Get date range
    start_idx, end_idx = date_idx
    selected_dates = date_range[start_idx:end_idx+1]
    
    # Filter dataframe to selected date range
    df_filtered = weekly_df.loc[selected_dates]
    
    # Apply moving average if window > 1
    if ma_window > 1:
        df_plot = df_filtered.rolling(window=ma_window, center=True).mean()
    else:
        df_plot = df_filtered
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Plot cases
    ax.plot(df_plot.index, df_plot['cases'], marker='o', 
            linestyle='-', linewidth=2, markersize=4, label='Weekly Cases')
    
    # Add trend line if checkbox is selected
    if show_trend and len(df_filtered) > 1:
        # Calculate linear trend
        x_numeric = np.arange(len(df_filtered))
        y_values = df_filtered['cases'].values
        # Only calculate trend if we have valid data
        if not np.all(y_values == 0):
            z = np.polyfit(x_numeric, y_values, 1)
            p = np.poly1d(z)
            ax.plot(df_filtered.index, p(x_numeric), "r--", 
                   linewidth=2, label='Trend Line')
    
    # Set scale
    if scale == 'log':
        ax.set_yscale('log')
    
    # Labels and formatting
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Weekly Cases', fontsize=12)
    ax.set_title('Measles Cases in England (Weekly)', fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def refresh_graph():
    """
    Force graphs to refresh by simulating widget interaction.
    This is called after API data refresh.
    """
    # Toggle scale to force redraw
    current = scale_dropdown.value
    scale_dropdown.value = 'log' if current == 'linear' else 'linear'
    scale_dropdown.value = current

# Create container for widgets
controls1 = wdg.VBox([
    date_slider,
    wdg.HBox([ma_slider, trend_checkbox, scale_dropdown])
])

# Link widgets to plotting function
graph1 = wdg.interactive_output(plot_timeseries, {
    'date_idx': date_slider,
    'ma_window': ma_slider,
    'show_trend': trend_checkbox,
    'scale': scale_dropdown
})

# Display widgets and graph
display(controls1, graph1)

## Graph 2: Age Group Distribution

This stacked bar chart shows measles case distribution across age groups. Use the controls to:
- **Time Period Dropdown**: Select which year or quarter to display
- **Aggregation Dropdown**: Choose monthly or quarterly aggregation
- **Age Group Selector**: Select which age groups to include in the chart

In [ ]:
# Create widgets for age distribution graph

# Get available years from data
years = sorted(age_df.index.year.unique().tolist())
year_options = [('All Years', 'all')] + [(str(y), y) for y in years]

# Year selector
year_selector = wdg.Dropdown(
    options=year_options,
    value='all',
    description='Time Period:',
    style={'description_width': 'initial'}
)

# Aggregation selector
aggregation_selector = wdg.Dropdown(
    options=[('Monthly', 'M'), ('Quarterly', 'Q'), ('Yearly', 'Y')],
    value='Q',
    description='Aggregation:',
    style={'description_width': 'initial'}
)

# Age group selector (multi-select)
age_groups = age_df.columns.tolist()
age_selector = wdg.SelectMultiple(
    options=age_groups,
    value=age_groups,
    description='Age Groups:',
    rows=min(8, len(age_groups)),
    style={'description_width': 'initial'}
)

def plot_age_distribution(selected_year, aggregation, selected_ages):
    """
    Plot age distribution of measles cases as stacked bar chart.
    
    Parameters:
    selected_year - year to filter (or 'all' for all years)
    aggregation - 'M' (monthly), 'Q' (quarterly), or 'Y' (yearly)
    selected_ages - list of age groups to include
    """
    # Filter by year if not 'all'
    if selected_year != 'all':
        df_filtered = age_df[age_df.index.year == selected_year]
    else:
        df_filtered = age_df
    
    # Filter by selected age groups
    if len(selected_ages) == 0:
        print("Please select at least one age group")
        return
    
    df_filtered = df_filtered[list(selected_ages)]
    
    # Aggregate data
    if aggregation == 'M':
        df_agg = df_filtered.groupby(pd.Grouper(freq='MS')).sum()
        date_format = '%Y-%m'
    elif aggregation == 'Q':
        df_agg = df_filtered.groupby(pd.Grouper(freq='QS')).sum()
        date_format = '%Y-Q%q'
    else:  # Yearly
        df_agg = df_filtered.groupby(pd.Grouper(freq='YS')).sum()
        date_format = '%Y'
    
    # Remove rows with all zeros
    df_agg = df_agg[(df_agg.T != 0).any()]
    
    if len(df_agg) == 0:
        print("No data available for selected filters")
        return
    
    # Create stacked bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot stacked bars
    df_agg.plot(kind='bar', stacked=True, ax=ax, 
                colormap='tab10', width=0.8)
    
    # Formatting
    ax.set_xlabel('Time Period', fontsize=12)
    ax.set_ylabel('Number of Cases', fontsize=12)
    ax.set_title('Measles Cases by Age Group', fontsize=14, fontweight='bold')
    
    # Format x-axis labels
    labels = [d.strftime(date_format) for d in df_agg.index]
    ax.set_xticklabels(labels, rotation=45, ha='right')
    
    # Legend outside plot area
    ax.legend(title='Age Group', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

# Create container for widgets
controls2 = wdg.HBox([
    wdg.VBox([year_selector, aggregation_selector]),
    age_selector
])

# Link widgets to plotting function
graph2 = wdg.interactive_output(plot_age_distribution, {
    'selected_year': year_selector,
    'aggregation': aggregation_selector,
    'selected_ages': age_selector
})

# Display widgets and graph
display(controls2, graph2)

## About This Dashboard

**Data Source:** Based on UK Government [data](https://ukhsa-dashboard.data.gov.uk/) published by the [UK Health Security Agency](https://www.gov.uk/government/organisations/uk-health-security-agency).

**Framework:** Based on the [DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) by Fabrizio Smeraldi.

**License:** Released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/).

**Dashboard Features:**
- Interactive time series visualization with customizable date ranges
- Age group distribution analysis with multiple aggregation options
- Real-time data refresh from UKHSA API
- Offline functionality with pre-loaded data

**Technical Implementation:**
- Data wrangling with pandas
- Visualization with matplotlib
- Interactive controls with ipywidgets
- API access with requests library